# 10. 거절 판정 개선 (v2.1): LLM이 "문서에 답이 있나?"를 먼저 판정

08 결과에서 **RAG 개선 + LoRA**는 답 없음 거절률 0.54, 대신 잘못 거절 0.102였습니다. Mac에서 08 답변 파일로 미리 확인한 내용:

| 확인 | 결과 |
|---|---|
| LoRA 잘못 거절 51개의 출처 | 점수 기준 20개 + **LLM이 스스로 거절 31개** (그중 16개는 정답 문단이 상위 5개에 없었음) |
| 재정렬 최고 점수로 답 있음/없음 구분 | AUC 0.80. 기준을 6.5로 올리면 답 없음 거절 0.54→0.67, 잘못 거절 0.10→**0.20** |
| LoRA가 거절하면 원본 Qwen 답 쓰기 | 잘못 거절 0.10→0.05, F1 0.349→0.359, 대신 답 없음 거절 0.54→**0.42** |

→ 점수 기준만으로는 한계가 있어서, **원본 Qwen에게 "이 문서들로 답할 수 있나? YES/NO"를 묻는 판정 단계**를 넣어 비교합니다.

- 08의 검색 결과·답변(`rag/eval_answers.jsonl`)을 그대로 쓰므로 **새로 생성하는 답은 없고**, 판정 점수만 계산합니다 (A100 또는 L4, 약 10분).
- 판정 기준 τ는 평가셋을 반으로 나눠 한쪽에서 고르고 다른 쪽에서 재는 **2겹 교차 검증**으로 고릅니다 (검증 200개에는 답 없는 질문이 없어서).

| 시스템 | 거절 규칙 | 답변 |
|---|---|---|
| S0: 08 그대로 | 점수 < 4.92 또는 LoRA가 거절 | LoRA |
| S1: 원본으로 대체 | 점수 < 4.92 또는 (LoRA·원본 둘 다 거절) | LoRA, LoRA가 거절하면 원본 Qwen |
| S2: S0 + 판정 | S0 규칙 또는 판정 P(YES) < τ | LoRA |
| **S3: 판정이 결정** | 점수 < 4.92 또는 판정 P(YES) < τ (판정 통과면 S1처럼 대체) | LoRA, 거절하면 원본 Qwen |

## 1. 드라이브 연결 + 경로

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'                 # v2.1 DB
RAG_DIR    = f'{SAVE_DIR}/rag'                                           # 08 결과
OUT_DIR    = f'{SAVE_DIR}/refusal'
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print('08 답변 파일 있음:', os.path.exists(f'{RAG_DIR}/eval_answers.jsonl'))

## 2. 설정 + 채점 함수 (08과 같음)

In [ ]:
import re, json, time, string, random
from collections import Counter
import numpy as np
import pandas as pd
import torch

SEED      = 42
LLM_NAME  = 'Qwen/Qwen2.5-7B-Instruct'
JUDGE_BATCH = 16
THRESH    = 4.921875     # 08 점수 기준 (검증 하위 5%)

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 또는 L4).'
print('GPU:', torch.cuda.get_device_name(0))

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def rouge_l(pred, ref):
    p, r = toks(pred), toks(ref)
    if not p or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for a in p:
        prev = 0
        for j, b in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if a == b else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

best = lambda fn, pred, refs: max(fn(pred, r) for r in refs)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

## 3. 08 결과 불러오기 + 숫자 재현 확인

In [ ]:
corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
ev     = {q['qid']: q for q in load_jsonl(f'{SAVE_DIR}/eval.jsonl')}
rows   = load_jsonl(f'{RAG_DIR}/eval_answers.jsonl')
for r in rows:
    q = ev[r['qid']]
    r['answers'], r['wf'] = q['answers'], q['well_formed']
    r['group'] = 'A' if r['domain'] == 'health' and r['answerable'] else ('N' if r['domain'] == 'health' else 'O')
    r['top'] = r['rag_docs'][0]['score']
    r['hit'] = bool(set(r['gold_pids'] or []) & {d['pid'] for d in r['rag_docs']})
print(Counter(r['group'] for r in rows))

A = [r for r in rows if r['group'] == 'A']
f1_lora = np.mean([best(f1, r['RAG 개선 + LoRA'], r['answers']) for r in A])
print(f'08 RAG 개선 + LoRA F1 재현: {f1_lora:.3f} (08 표: 0.349)')
assert abs(f1_lora - 0.349) < 0.002, '08 결과 파일이 표와 다릅니다'

## 4. 판정: 원본 Qwen에게 "문서에 답이 있나?" (YES/NO 확률)

답을 생성하지 않고 첫 토큰 확률만 봅니다. P(YES) = YES 계열 토큰 확률 / (YES 계열 + NO 계열). 결과는 `refusal/judge_scores.jsonl`에 저장하고, 있으면 불러옵니다.

In [ ]:
JUDGE_PATH = f'{OUT_DIR}/judge_scores.jsonl'
SYS_JUDGE = ('You check whether numbered documents contain the information needed to answer a medical question. '
             'Reply with only YES or NO.')

def judge_messages(r):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {texts[d["pid"]]}' for i, d in enumerate(r['rag_docs'], 1))
    return [{'role': 'system', 'content': SYS_JUDGE},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {r["query"]}\n\n'
                                        'Do the documents contain the answer to this question? Reply YES or NO.'}]

if os.path.exists(JUDGE_PATH):
    saved = {j['qid']: j['p_yes'] for j in load_jsonl(JUDGE_PATH)}
    print('저장된 판정 불러옴:', len(saved))
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda').eval()
    first = lambda w: tok.encode(w, add_special_tokens=False)[0]
    YES = sorted({first(w) for w in ['YES', 'Yes', 'yes']})
    NO  = sorted({first(w) for w in ['NO', 'No', 'no']})
    print('YES 토큰', YES, '/ NO 토큰', NO)

    saved, t = {}, time.time()
    with torch.inference_mode():
        for i in range(0, len(rows), JUDGE_BATCH):
            batch = rows[i:i + JUDGE_BATCH]
            prompts = [tok.apply_chat_template(judge_messages(r), tokenize=False, add_generation_prompt=True) for r in batch]
            enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
            logits = llm(**enc).logits[:, -1].float()
            p = torch.softmax(logits, dim=-1)
            p_yes = p[:, YES].sum(-1) / (p[:, YES].sum(-1) + p[:, NO].sum(-1))
            for r, v in zip(batch, p_yes.tolist()):
                saved[r['qid']] = v
    print(f'판정 끝: {len(saved)}개, {time.time() - t:.0f}초')
    save_jsonl([{'qid': k, 'p_yes': v} for k, v in saved.items()], JUDGE_PATH)
    del llm; torch.cuda.empty_cache()

for r in rows:
    r['p_yes'] = saved[r['qid']]

## 5. 답 있음 vs 답 없음 구분력 (AUC, 높을수록 잘 가름)

In [ ]:
from sklearn.metrics import roc_auc_score
AN = [r for r in rows if r['group'] in 'AN']
y = [r['group'] == 'A' for r in AN]
auc = {'재정렬 최고 점수': roc_auc_score(y, [r['top'] for r in AN]),
       '판정 P(YES)':     roc_auc_score(y, [r['p_yes'] for r in AN])}
# 정답 문단이 상위 5개에 있는 질문(답 있음) vs 답 없음: 판정이 '근거 유무'를 보는지
AN_hit = [r for r in AN if r['group'] == 'N' or r['hit']]
auc['판정 P(YES), 정답 문단 있는 질문만'] = roc_auc_score([r['group'] == 'A' for r in AN_hit], [r['p_yes'] for r in AN_hit])
auc_table = pd.Series(auc).round(3).to_frame('AUC')
display(auc_table)
for g in 'ANO':
    v = [r['p_yes'] for r in rows if r['group'] == g]
    print(g, '평균 P(YES)', round(float(np.mean(v)), 3), '/ 0.5 미만 비율', round(float(np.mean(np.array(v) < 0.5)), 3))

## 6. 시스템 비교

S2·S3의 τ는 **2겹 교차 검증**: 그룹(A/N/O)별로 섞어 반으로 나누고, 한쪽에서 τ를 골라 다른 쪽에 적용합니다. 두 번 해서 합친 결과를 보고합니다.\n\n- **균형**: "답 없음 거절률 − 잘못 거절"이 가장 큰 τ\n- **제한**: 잘못 거절 ≤ 0.10 (08 수준)인 τ 중 답 없음 거절률이 가장 큰 τ

In [ ]:
LORA, BASE = 'RAG 개선 + LoRA', 'RAG 개선'

def answer_of(r, system, tau=None):
    if r['top'] < THRESH:
        return REFUSAL
    lora, base = r[LORA], r[BASE]
    if system == 'S0':
        return lora
    if system == 'S1':
        return base if is_refusal(lora) else lora
    if r['p_yes'] < tau:
        return REFUSAL
    if system == 'S2':
        return lora
    if system == 'S3':
        return base if is_refusal(lora) else lora

def metrics(rs, answers):
    by = lambda g: [(r, a) for r, a in zip(rs, answers) if r['group'] == g]
    A_, N_, O_ = by('A'), by('N'), by('O')
    wf = [(r, a) for r, a in A_ if r['wf']]
    return {
        'F1':                  np.mean([best(f1, a, r['answers']) for r, a in A_]),
        'ROUGE-L 문장형':      np.mean([best(rouge_l, a, r['wf']) for r, a in wf]) if wf else np.nan,
        '잘못 거절':           np.mean([is_refusal(a) for r, a in A_]),
        '잘못 거절 (정답 문단 있음)': np.mean([is_refusal(a) for r, a in A_ if r['hit']]),
        '정답 문단 없을 때 거절': np.mean([is_refusal(a) for r, a in A_ if not r['hit']]),
        '답 없음 거절률':      np.mean([is_refusal(a) for r, a in N_]),
        '분야 밖 거절률':      np.mean([is_refusal(a) for r, a in O_]),
    }

TAUS = [round(float(x), 2) for x in np.arange(0.05, 0.96, 0.05)]
CAP = 0.10   # '제한' 방식: 잘못 거절이 08(0.102) 수준 이하인 τ 중에서 답 없음 거절률 최대

def pick_tau(rs, system, mode):
    ms = {tau: metrics(rs, [answer_of(r, system, tau) for r in rs]) for tau in TAUS}
    if mode == '균형':
        return max(TAUS, key=lambda t: ms[t]['답 없음 거절률'] - ms[t]['잘못 거절'])
    ok = [t for t in TAUS if ms[t]['잘못 거절'] <= CAP] or [TAUS[0]]
    return max(ok, key=lambda t: (ms[t]['답 없음 거절률'], -ms[t]['잘못 거절']))

# 그룹별로 섞어 2겹으로 나누기
rng = random.Random(SEED)
fold = {}
for g in 'ANO':
    ids = [r['qid'] for r in rows if r['group'] == g]
    rng.shuffle(ids)
    for k, q in enumerate(ids):
        fold[q] = k % 2

results, chosen = {}, {}
results['S0: 08 그대로'] = metrics(rows, [answer_of(r, 'S0') for r in rows])
results['S1: 원본으로 대체'] = metrics(rows, [answer_of(r, 'S1') for r in rows])
for system, label in [('S2', 'S2: S0 + 판정'), ('S3', 'S3: 판정이 결정')]:
    for mode in ('균형', '제한'):
        ans, taus = {}, []
        for k in (0, 1):
            tr = [r for r in rows if fold[r['qid']] != k]
            te = [r for r in rows if fold[r['qid']] == k]
            tau = pick_tau(tr, system, mode)
            taus.append(tau)
            for r in te:
                ans[r['qid']] = answer_of(r, system, tau)
        results[f'{label}, {mode}'] = metrics(rows, [ans[r['qid']] for r in rows])
        chosen[f'{system} {mode}'] = {'교차 검증 τ': taus, '전체로 고른 τ (챗봇용)': pick_tau(rows, system, mode)}

table = pd.DataFrame(results).T.round(3)
display(table)
print('선택된 τ:', chosen)

## 7. τ에 따른 변화 (S3, 전체 650개) — 발표용 트레이드오프 표

In [ ]:
sweep = pd.DataFrame({tau: metrics(rows, [answer_of(r, 'S3', tau) for r in rows]) for tau in TAUS}).T.round(3)
sweep.index.name = 'τ'
display(sweep[['F1', '잘못 거절', '잘못 거절 (정답 문단 있음)', '답 없음 거절률', '분야 밖 거절률']])

In [ ]:
# 판정이 바꾼 사례 (S0 → S3, 챗봇용 τ): 발표·사례 분석용
TAU = chosen['S3 제한']['전체로 고른 τ (챗봇용)']
def show(r):
    print(f"\n[{r['group']}] P(YES)={r['p_yes']:.2f} 점수={r['top']:.2f} 정답문단={'있음' if r['hit'] else '없음'}  {r['query']}")
    print('  S0:', r[LORA][:120]); print('  S3:', answer_of(r, 'S3', TAU)[:120])
newly_refused_N = [r for r in rows if r['group'] == 'N' and not is_refusal(answer_of(r, 'S0')) and is_refusal(answer_of(r, 'S3', TAU))]
rescued_A = [r for r in rows if r['group'] == 'A' and is_refusal(answer_of(r, 'S0')) and not is_refusal(answer_of(r, 'S3', TAU))]
print(f'답 없음인데 새로 거절: {len(newly_refused_N)}개 / 잘못 거절이었다가 답함: {len(rescued_A)}개')
for r in newly_refused_N[:3] + rescued_A[:3]:
    show(r)

## 8. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
summary = '\n'.join([
    f'# 10 거절 판정 개선 (v2.1) ({stamp} KST)', '',
    f'- 08 검색 결과·답변 재사용 (답 있는 건강 500 + 답 없음 100 + 분야 밖 50), 점수 기준 {THRESH:.2f} 유지',
    f'- 판정: {LLM_NAME} (LoRA 없음)에게 "문서에 답이 있나?" YES/NO 첫 토큰 확률',
    '- S2·S3의 τ: 2겹 교차 검증. 균형 = "답 없음 거절률 − 잘못 거절" 최대 / 제한 = 잘못 거절 ≤ 0.10 중 답 없음 거절률 최대',
    f'- 선택된 τ: {json.dumps(chosen, ensure_ascii=False)}', '',
    '## 답 있음 vs 답 없음 구분력', '', auc_table.to_markdown(), '',
    '## 시스템 비교', '', table.to_markdown(), '',
    '## τ에 따른 변화 (S3, 전체)', '', sweep[['F1', '잘못 거절', '잘못 거절 (정답 문단 있음)', '답 없음 거절률', '분야 밖 거절률']].to_markdown(),
])
for d in (OUT_DIR, REPORT_DIR):
    path = f'{d}/results_summary.md' if d == OUT_DIR else f'{d}/10_refusal_summary.md'
    with open(path, 'w', encoding='utf-8') as f:
        f.write(summary)
with open(f'{OUT_DIR}/results.json', 'w', encoding='utf-8') as f:
    json.dump({'thresh': THRESH, 'chosen': chosen, 'auc': auc, 'table': table.to_dict(orient='index')}, f, ensure_ascii=False, indent=2)
print(summary)

## 9. 드라이브 저장 마무리 + 런타임 종료

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()